# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AnanyaGujral/flyrank-ml-internship-start/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

Runs offline on the bundled slice. This is the transparent rule every model must beat.

## 1. My rule and its reason codes

**The rule, in plain words:** *a page is worth reviewing when it is visible but under-earning clicks —
prior-window CTR is below half the pool median — and the bigger its prior-window exposure, the higher
it ranks.* Nothing else. No learned weights.

**Reason codes it can output (feature-only, never label-derived):**
- `ctr_gap` — prior-window CTR at or below half the pool median;
- `high_exposure` — prior-window impressions in the top quartile of the pool;
- `no_clicks` — prior-window clicks are zero despite real exposure.

In [1]:
# ---- shared offline setup: bundled slice, leak-free pool + features, transparent rule ----
import json
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
MIN_PREV_IMPRESSIONS = 100     # "already visible" floor on the TRAILING window
DROP_FACTOR = 0.8              # declined = next-30 impressions < 80% of prior-30

root = Path.cwd().resolve()
while root != root.parent and not (root / "AGENTS.md").exists():
    root = root.parent
DATA = root / "data" / "raw" / "content_refresh_anonymized.csv"
OUT = root / "work" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
raw = pd.read_csv(DATA)


def build_pool(frame):
    """Already-visible pages with a usable trailing window, plus leak-free features.

    Every feature is measured in the PRIOR 30-day window (days 31-60) or is static
    metadata. Nothing is taken from the 90-day totals or the outcome window, because
    those overlap the label's comparison window.
    """
    pool = frame[frame["impressions_prev_30d"] >= MIN_PREV_IMPRESSIONS].copy()
    pool["declined_next_30d"] = (
        pool["impressions_last_30d"] < DROP_FACTOR * pool["impressions_prev_30d"]).astype(int)
    pool["log_impr_prev"] = np.log1p(pool["impressions_prev_30d"])
    pool["log_clk_prev"] = np.log1p(pool["clicks_prev_30d"])
    pool["log_sess_prev"] = np.log1p(pool["sessions_prev_30d"])
    pool["ctr_prev"] = 100 * pool["clicks_prev_30d"] / pool["impressions_prev_30d"]
    pool["clk_per_sess_prev"] = pool["clicks_prev_30d"] / pool["sessions_prev_30d"].replace(0, np.nan)
    pool["has_keyword"] = pool["search_volume"].notna().astype(int)
    pool["has_clicks_prev"] = (pool["clicks_prev_30d"] > 0).astype(int)
    pool["has_sessions_prev"] = (pool["sessions_prev_30d"] > 0).astype(int)
    g = pool.groupby("client_id")
    pool["ctr_prev_z"] = ((pool["ctr_prev"] - g["ctr_prev"].transform("mean"))
                          / g["ctr_prev"].transform("std").replace(0, np.nan))
    pool["impr_prev_share"] = pool["impressions_prev_30d"] / g["impressions_prev_30d"].transform("sum")
    pool["clk_prev_share"] = pool["clicks_prev_30d"] / g["clicks_prev_30d"].transform("sum").replace(0, np.nan)
    return pool


NUM_FEATURES = ["log_impr_prev", "log_clk_prev", "log_sess_prev", "ctr_prev", "clk_per_sess_prev",
                "ctr_prev_z", "impr_prev_share", "clk_prev_share",
                "search_volume", "competition", "cpc", "word_count", "char_count",
                "content_age_days", "days_since_last_update",
                "has_keyword", "has_clicks_prev", "has_sessions_prev"]
CAT_FEATURES = ["competition_level", "content_type", "main_intent", "age_tier",
                "freshness_tier", "word_count_tier"]


def feature_matrix(pool):
    Xn = pool[NUM_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return pd.concat([Xn, pd.get_dummies(pool[CAT_FEATURES].astype(str), dtype=float)], axis=1)


def rulescore(pool):
    """Transparent rule: low trailing CTR, weighted by trailing exposure."""
    med = float(pool["ctr_prev"].median())
    return (pool["ctr_prev"] <= 0.5 * med).to_numpy() * np.log1p(pool["impressions_prev_30d"].to_numpy())


pool = build_pool(raw)
X = feature_matrix(pool)
y = pool["declined_next_30d"].to_numpy()
groups = pool["client_id"].to_numpy()
rule = rulescore(pool)
print(f"bundled slice: {len(raw):,} rows -> pool {len(pool):,} pages | "
      f"{pool['client_id'].nunique()} pseudo-clients")
print(f"decline base rate in pool: {y.mean():.3f} | feature matrix: {X.shape}")
print("label: next-30 impressions < 80% of prior-30 (all features from the prior-30 window + metadata)")


def pk(table_scores, y_true, ks=(10, 20, 50, 100)):
    y_true = np.asarray(y_true)
    order = np.argsort(-np.asarray(table_scores))
    return {str(k): round(float(y_true[order[:k]].mean()), 3) for k in ks}

bundled slice: 30,000 rows -> pool 18,010 pages | 30 pseudo-clients
decline base rate in pool: 0.616 | feature matrix: (18010, 43)
label: next-30 impressions < 80% of prior-30 (all features from the prior-30 window + metadata)


In [2]:
def pk(table_scores, y_true, ks=(10, 20, 50, 100)):
    y_true = np.asarray(y_true)
    order = np.argsort(-np.asarray(table_scores))
    return {str(k): round(float(y_true[order[:k]].mean()), 3) for k in ks}

In [3]:
median_ctr = float(pool["ctr_prev"].median())
exposure_q75 = float(pool["impressions_prev_30d"].quantile(0.75))
ctr_gap = pool["ctr_prev"] <= 0.5 * median_ctr
high_exposure = pool["impressions_prev_30d"] >= exposure_q75
no_clicks = (pool["clicks_prev_30d"] == 0) & (pool["impressions_prev_30d"] >= 500)

def reason_code(i):
    codes = []
    if ctr_gap.iloc[i]:
        codes.append("ctr_gap")
    if no_clicks.iloc[i]:
        codes.append("no_clicks")
    if high_exposure.iloc[i]:
        codes.append("high_exposure")
    return "+".join(codes) if codes else "base_anchor"

pool["rule_score"] = rule
pool["reason_code"] = [reason_code(i) for i in range(len(pool))]
print(f"pool median prior CTR: {median_ctr:.3f} | exposure top-quartile floor: {exposure_q75:,.0f} impressions")
print(f"rule fires on {int((rule > 0).sum()):,} of {len(pool):,} pages ({100*(rule > 0).mean():.1f}%)")

pool median prior CTR: 0.114 | exposure top-quartile floor: 2,471 impressions
rule fires on 7,637 of 18,010 pages (42.4%)


## 2. Build the ranked queue (writes the CSV)

Rank every pool page by the rule, attach the reason code, and write the queue. The CSV lands in
`work/outputs/` (gitignored — datasets never enter git).

In [4]:
queue = pool.sort_values("rule_score", ascending=False).reset_index(drop=True)
queue_out = queue[["content_id", "client_id", "rule_score", "reason_code",
                   "impressions_prev_30d", "clicks_prev_30d", "ctr_prev",
                   "days_since_last_update", "declined_next_30d"]].copy()
queue_out.insert(0, "rank", np.arange(1, len(queue_out) + 1))
csv_path = OUT / "baseline_action_score.csv"
queue_out.to_csv(csv_path, index=False)
print(f"queue written: {csv_path.name} | rows {len(queue_out):,} | gitignored under work/**/*.csv")

queue written: baseline_action_score.csv | rows 18,010 | gitignored under work/**/*.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, a confidence note, and what would make it wrong.*

In [5]:
top = queue_out.head(20)
print(f"{'rank':>4} {'reason':<24} {'prev impr':>10} {'prev clk':>9} {'ctr%':>6} {'stale d':>8} {'outcome':>8}")
for _, r in top.iterrows():
    print(f"{int(r['rank']):>4} {r['reason_code']:<24} {int(r['impressions_prev_30d']):>10,} "
          f"{int(r['clicks_prev_30d']):>9,} {r['ctr_prev']:>6.3f} {int(r['days_since_last_update']):>8} "
          f"{'DECLINED' if r['declined_next_30d'] else 'held':>8}")
print()
print("Confidence note: high when ctr_gap AND high_exposure both fire (the page is big and under-earning).")
print("What would make it wrong: a deliberate low-CTR intent (e.g. a brand/definitional query), a")
print("one-off campaign window, or a page whose real problem is position loss the rule cannot see.")

rank reason                    prev impr  prev clk   ctr%  stale d  outcome
   1 ctr_gap+no_clicks+high_exposure    111,885         0  0.000      104 DECLINED
   2 ctr_gap+high_exposure       110,679        61  0.055      104     held
   3 ctr_gap+high_exposure        97,606        30  0.031       20     held
   4 ctr_gap+high_exposure        96,122        37  0.038       20 DECLINED
   5 ctr_gap+high_exposure        94,762        47  0.050      104 DECLINED
   6 ctr_gap+high_exposure        84,773        19  0.022       20     held
   7 ctr_gap+high_exposure        75,160        31  0.041      104     held
   8 ctr_gap+high_exposure        69,303        15  0.022       20 DECLINED
   9 ctr_gap+high_exposure        65,928        19  0.029      104 DECLINED
  10 ctr_gap+high_exposure        64,917         6  0.009       20 DECLINED
  11 ctr_gap+high_exposure        61,139        33  0.054       25     held
  12 ctr_gap+high_exposure        54,506        13  0.024       20 DECLINED
  13 

## 4. Weak picks + leakage check

Which picks look wrong, and did anything from the outcome window leak into the score?

In [6]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

held_out = pk(rule, y)
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42).split(X, y, groups))
rule_heldout = pk(rule[te], y[te])

weak = queue.head(200)
weak_false = weak[weak["declined_next_30d"] == 0]
print(f"rule precision@K, WHOLE pool (honest-ish but optimistic): {held_out}")
print(f"rule precision@K, HELD-OUT clients (the number the model must beat): {rule_heldout}")
print(f"held-out base rate: {y[te].mean():.3f} | held-out rule ROC-AUC: "
      f"{roc_auc_score(y[te], rule[te]):.3f}")
print(f"\nweak picks: {len(weak_false)} of the top 200 did NOT decline next window")
print("the leakage check: rule inputs are prior-30 columns only ->",
      "PASS" if not any(c.endswith('_last_30d') for c in ['impressions_prev_30d', 'ctr_prev']) else "FAIL")

rule precision@K, WHOLE pool (honest-ish but optimistic): {'10': 0.6, '20': 0.75, '50': 0.8, '100': 0.78}
rule precision@K, HELD-OUT clients (the number the model must beat): {'10': 0.7, '20': 0.7, '50': 0.76, '100': 0.67}
held-out base rate: 0.567 | held-out rule ROC-AUC: 0.580

weak picks: 46 of the top 200 did NOT decline next window
the leakage check: rule inputs are prior-30 columns only -> PASS


In [7]:
metrics = {
    "task": "ml-07",
    "data": "data/raw/content_refresh_anonymized.csv",
    "pool_rows": int(len(pool)),
    "base_rate": round(float(y.mean()), 3),
    "rule": "score = (ctr_prev <= 0.5 * pool_median_ctr) * log1p(impressions_prev_30d)",
    "median_ctr": round(median_ctr, 4),
    "precision_at_k_whole_pool": held_out,
    "precision_at_k_heldout_clients": rule_heldout,
    "heldout_base_rate": round(float(y[te].mean()), 3),
    "heldout_roc_auc": round(float(roc_auc_score(y[te], rule[te])), 3),
    "reason_codes": ["ctr_gap", "no_clicks", "high_exposure", "base_anchor"],
}
(OUT / "w04_baseline_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print("receipt written: work/outputs/w04_baseline_metrics.json")

receipt written: work/outputs/w04_baseline_metrics.json


### Self-check

- [x] Rule stated in plain words before any code
- [x] Reason codes are feature-only and human-readable
- [x] Queue written (to a gitignored path)
- [x] Top-20 reviewed with a "what would make it wrong" line
- [x] Held-out precision@K reported next to the base rate, leakage checked
- [x] Committed under `work/notebooks/`